# Workshop: Optimisation and Numerical Stability

This notebook contains **two short workshop tasks** connected to the lecture topics on:

- gradient descent  
- global minimum vs local minimum  
- unstable optimisation  
- learning rate and feature scaling  
- logits vs probabilities  
- overflow and numerical stability in softmax  

Use this notebook in **Google Colab**. Run each section carefully and answer the short questions in the markdown cells.

## Learning goals

By the end of this workshop, you should be able to:

1. recognise unstable optimisation during training  
2. improve convergence by tuning optimisation settings  
3. explain why a simple convex loss has a global minimum  
4. distinguish between **logits** and **probabilities**  
5. explain why naïve softmax can become numerically unstable

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)

# Task 1: Fix an Unstable Optimisation Process

## Problem statement

You are given a simple linear regression model:

\[
\hat{y} = wx + b
\]

The data follow a linear relationship, so the optimisation problem is **convex**.  
That means there is a **single global minimum** for the mean squared error loss.

However, the training setup below is intentionally unstable. Your job is to make the optimisation converge properly.

## Your task

1. Run the baseline training code.  
2. Observe the loss curve and parameter updates.  
3. Improve convergence by changing:
   - the learning rate
   - the number of epochs
   - the feature scaling (recommended)
4. Try to obtain parameters close to the true values:
   - true slope = **3**
   - true intercept = **2**

## Questions

- What happens when the learning rate is too large?  
- Why is this problem expected to have a global minimum?  
- How does feature scaling affect optimisation?

In [ ]:
# Synthetic dataset
X = np.linspace(-50, 50, 200).reshape(-1, 1)
y = 3 * X + 2

print("X shape:", X.shape)
print("y shape:", y.shape)

plt.scatter(X, y, s=12)
plt.title("Synthetic regression data")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

In [ ]:
def train_linear_regression(X, y, w_init=10.0, b_init=20.0, learning_rate=0.5, epochs=50, verbose=True):
    '''
    Trains a 1D linear regression model using gradient descent.

    Parameters:
        X: input array of shape (n, 1)
        y: target array of shape (n, 1) or (n,)
        w_init: initial weight
        b_init: initial bias
        learning_rate: gradient descent step size
        epochs: number of iterations
        verbose: whether to print progress

    Returns:
        w, b, losses
    '''
    X = X.astype(float)
    y = y.reshape(-1, 1).astype(float)

    w = np.array([[w_init]], dtype=float)
    b = np.array([[b_init]], dtype=float)
    losses = []

    for epoch in range(epochs):
        y_pred = X @ w + b
        error = y_pred - y

        loss = np.mean(error ** 2)
        losses.append(loss)

        dw = (2 / len(X)) * np.sum(error * X)
        db = (2 / len(X)) * np.sum(error)

        w = w - learning_rate * dw
        b = b - learning_rate * db

        if verbose and (epoch < 10 or epoch % 10 == 9):
            print(f"Epoch {epoch+1:3d} | loss = {loss:12.4f} | w = {w[0,0]:10.4f} | b = {b[0,0]:10.4f}")

    return w[0, 0], b[0, 0], losses

## Baseline run

The following settings are intentionally poor.  
Run the cell and observe whether the loss decreases smoothly or becomes unstable.

In [ ]:
w_bad, b_bad, losses_bad = train_linear_regression(
    X, y,
    w_init=10.0,
    b_init=20.0,
    learning_rate=0.5,   # intentionally too large
    epochs=50,
    verbose=True
)

plt.plot(losses_bad)
plt.title("Baseline training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.show()

print("Final parameters from baseline:")
print("w =", w_bad)
print("b =", b_bad)

## Student activity

Now create a better training setup.

### Suggestions
- Try a smaller learning rate, such as `0.1`, `0.01`, or `0.001`
- Try more epochs
- Standardise the input feature `X`
- Compare the new loss curve with the baseline loss curve

Complete the code below.

In [ ]:
# TODO: Try improved settings here

# Example: feature scaling
X_scaled = (X - X.mean()) / X.std()

w_good, b_good, losses_good = train_linear_regression(
    X_scaled, y,
    w_init=0.0,
    b_init=0.0,
    learning_rate=0.05,   # change this if needed
    epochs=200,
    verbose=False
)

plt.plot(losses_good)
plt.title("Improved training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.show()

print("Final parameters (scaled-input training):")
print("w =", w_good)
print("b =", b_good)

## Reflection questions for Task 1

Write short answers in a text cell below:

1. What was wrong with the original setup?  
2. Which changes improved convergence?  
3. Why can we talk about a **global minimum** in this task?  
4. Did scaling the input help? Why?

# Task 2: Logits, Probabilities, and Numerically Stable Softmax

## Problem statement

In classification, models often produce **logits**.

- **Logits** are raw scores from the model.
- **Probabilities** are obtained after applying **softmax**.

A naïve softmax implementation can become unstable when logits are very large, because:

\[
e^{1000}
\]

is too large for normal floating-point computation.

Your task is to compare:
1. a naïve softmax  
2. a numerically stable softmax  

and explain the difference.

In [ ]:
large_logits = np.array([1000.0, 1001.0, 1002.0])

def naive_softmax(x):
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)

def stable_softmax(x):
    x_shifted = x - np.max(x)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x)

print("Large logits:", large_logits)

In [ ]:
print("Naive softmax:")
try:
    print(naive_softmax(large_logits))
except Exception as e:
    print("Error:", e)

print("\nStable softmax:")
print(stable_softmax(large_logits))

## What to notice

The naïve implementation may produce:
- `inf`
- `nan`
- runtime warnings

The stable implementation works by subtracting the maximum logit before exponentiation:

\[
\text{softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}
\]

This does **not** change the final probabilities, but it improves numerical stability.

In [ ]:
# Smaller logits for interpretation
logits = np.array([2.0, 1.0, 0.1])
probs = stable_softmax(logits)

print("Logits:", logits)
print("Probabilities:", probs)
print("Predicted class index:", np.argmax(probs))
print("Sum of probabilities:", probs.sum())

## Student activity

Answer these questions in a text cell below:

1. What is the difference between **logits** and **probabilities**?  
2. Why can the naïve softmax overflow?  
3. Why does subtracting the maximum value improve stability?  
4. Why do many deep learning libraries compute cross-entropy directly from logits rather than probabilities?

# Optional Extension

Try changing the logits in the code above.

Examples:
- `[10, 11, 12]`
- `[100, 101, 102]`
- `[-1000, -1001, -1002]`

Observe what happens in the naïve and stable implementations.

# Submission checklist

For this workshop, submit:

## Task 1
- the baseline loss plot  
- the improved loss plot  
- your final parameter values  
- short answers to the reflection questions  

## Task 2
- the naïve softmax output  
- the stable softmax output  
- short explanations of:
  - logits vs probabilities
  - overflow
  - numerical stability

# Instructor note

Task 1 uses a **convex** regression problem, so it is appropriate to discuss a **global minimum**.  
Task 2 links the lecture's optimisation content to practical implementation issues such as **overflow** and **stable softmax**.